# End-to-End Stochastic Computing Pipeline

**SC-NeuroCore v3.8** — From bitstream encoding through synaptic computation to spiking neuron output.

This notebook walks through the complete SC inference pipeline:

1. **Encode** analog values as stochastic bitstreams
2. **Multiply** via AND-gate synapses (BitstreamSynapse)
3. **Accumulate** via popcount dot-product (BitstreamDotProduct)
4. **Integrate** current in a leaky integrate-and-fire neuron
5. **Scale up** to a VectorizedSCLayer for multi-neuron inference

> © 1998–2026 Miroslav Šotek. All rights reserved.  
> License: GNU AFFERO GENERAL PUBLIC LICENSE v3 | Commercial Licensing Available  
> Contact: www.anulum.li | protoscience@anulum.li

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from sc_neurocore import (
    BitstreamEncoder,
    bitstream_to_probability,
    BitstreamSynapse,
    BitstreamDotProduct,
    StochasticLIFNeuron,
    VectorizedSCLayer,
)

np.random.seed(42)
print("SC-NeuroCore end-to-end pipeline demo")

## 1. Bitstream Encoding

A stochastic bitstream represents a probability $p \in [0, 1]$ as a sequence of
random bits where $\Pr(\text{bit}=1) = p$. The LFSR (16-bit maximal-length,
polynomial $x^{16}+x^{14}+x^{13}+x^{11}+1$) provides decorrelated pseudo-random
thresholds for each encoder.

In [ ]:
LENGTH = 1024
target_p = 0.7

encoder = BitstreamEncoder(x_min=0.0, x_max=1.0, length=LENGTH, seed=0xACE1)
bitstream = encoder.encode(target_p)

recovered_p = bitstream_to_probability(bitstream)
print(f"Target: {target_p:.3f}  Recovered: {recovered_p:.3f}  "
      f"Error: {abs(recovered_p - target_p):.4f}")

fig, axes = plt.subplots(1, 2, figsize=(12, 3))
axes[0].step(range(100), bitstream[:100], where="post", linewidth=0.8)
axes[0].set_title(f"First 100 bits (p={target_p})")
axes[0].set_xlabel("Bit index")
axes[0].set_ylabel("Value")
axes[0].set_ylim(-0.1, 1.1)

ps = np.linspace(0, 1, 21)
recovered = [bitstream_to_probability(encoder.encode(p)) for p in ps]
axes[1].plot(ps, recovered, "o-", markersize=4)
axes[1].plot([0, 1], [0, 1], "k--", alpha=0.3)
axes[1].set_title(f"Encode/Decode Linearity (L={LENGTH})")
axes[1].set_xlabel("Target probability")
axes[1].set_ylabel("Recovered probability")
plt.tight_layout()
plt.show()

## 2. Synaptic Multiplication (AND Gate)

In stochastic computing, multiplication is a single AND gate:

$$\Pr(x \wedge w = 1) = \Pr(x=1) \cdot \Pr(w=1) = p_x \cdot p_w$$

A `BitstreamSynapse` encodes its weight $w$ as a bitstream and ANDs it
with the pre-synaptic input bitstream.

In [ ]:
weights = [0.3, 0.6, 0.9]
input_p = 0.8

enc = BitstreamEncoder(x_min=0.0, x_max=1.0, length=LENGTH, seed=0xACE1)
input_bits = enc.encode(input_p)

for w in weights:
    syn = BitstreamSynapse(w_min=0.0, w_max=1.0, w=w, seed=0xBEEF, length=LENGTH)
    out_bits = syn.apply(input_bits)
    product = bitstream_to_probability(out_bits)
    expected = input_p * w
    print(f"  x={input_p:.1f} * w={w:.1f} = {product:.3f}  "
          f"(expected {expected:.3f}, err={abs(product-expected):.4f})")

## 3. Dot Product via Popcount

A `BitstreamDotProduct` combines multiple synapses. For input vector
$\mathbf{x}$ and weight vector $\mathbf{w}$:

$$I = \sum_i x_i \cdot w_i$$

Each synapse ANDs its input bitstream with its weight bitstream, then
popcount across all synapses yields the fixed-point current $I$.

In [ ]:
N_INPUTS = 4
x_vals = [0.8, 0.4, 0.6, 0.3]
w_vals = [0.5, 0.7, 0.2, 0.9]

synapses = [
    BitstreamSynapse(w_min=0.0, w_max=1.0, w=w, seed=0xBEEF + i * 13, length=LENGTH)
    for i, w in enumerate(w_vals)
]
dp = BitstreamDotProduct(synapses)

encoders = [
    BitstreamEncoder(x_min=0.0, x_max=1.0, length=LENGTH, seed=0xACE1 + i * 7)
    for i in range(N_INPUTS)
]
pre_matrix = np.stack([enc.encode(x) for enc, x in zip(encoders, x_vals)])

out_bits, current = dp.apply(pre_matrix)
expected_dot = sum(x * w for x, w in zip(x_vals, w_vals))
print(f"SC dot product:  {current:.4f}")
print(f"Exact dot product: {expected_dot:.4f}")
print(f"Absolute error:  {abs(current - expected_dot):.4f}")

## 4. Leaky Integrate-and-Fire Neuron

The `StochasticLIFNeuron` integrates synaptic current $I(t)$ with
exponential leak:

$$\tau_m \frac{dV}{dt} = -(V - V_{\text{rest}}) + R \cdot I(t)$$

When $V \geq V_{\text{thresh}}$, the neuron fires and resets.

In [ ]:
neuron = StochasticLIFNeuron(v_threshold=0.8, tau_mem=10.0, dt=1.0)

N_STEPS = 200
voltages = []
spikes = []

for t in range(N_STEPS):
    pre = np.stack([enc.encode(x) for enc, x in zip(encoders, x_vals)])
    _, I = dp.apply(pre)
    spike = neuron.step(I)
    voltages.append(neuron.v)
    spikes.append(int(spike))

fig, axes = plt.subplots(2, 1, figsize=(12, 5), sharex=True)
axes[0].plot(voltages, linewidth=0.8)
axes[0].axhline(0.8, color="r", linestyle="--", alpha=0.5, label="threshold")
axes[0].set_ylabel("Membrane voltage")
axes[0].legend()
axes[0].set_title("LIF Neuron Response to SC Dot Product Current")

spike_times = [t for t, s in enumerate(spikes) if s]
axes[1].eventplot([spike_times], linewidths=0.8)
axes[1].set_ylabel("Spikes")
axes[1].set_xlabel("Time step")
rate = sum(spikes) / N_STEPS * 1000
axes[1].set_title(f"Spike raster ({sum(spikes)} spikes, ~{rate:.0f} Hz equivalent)")
plt.tight_layout()
plt.show()

## 5. Scaling Up: VectorizedSCLayer

`VectorizedSCLayer` parallelises the encode→synapse→accumulate pipeline
across multiple neurons using NumPy (or CuPy on GPU). Bitstream operations
become packed-word AND + popcount, yielding ~58 Gbit/s on CPU.

In [ ]:
N_IN, N_OUT, L = 8, 4, 512
layer = VectorizedSCLayer(n_inputs=N_IN, n_neurons=N_OUT, length=L)

inputs = np.array([0.1, 0.3, 0.5, 0.7, 0.9, 0.2, 0.4, 0.6])
N_TRIALS = 100
outputs = np.array([layer.forward(inputs) for _ in range(N_TRIALS)])

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for n in range(N_OUT):
    axes[0].plot(outputs[:, n], alpha=0.7, label=f"Neuron {n}")
axes[0].set_xlabel("Trial")
axes[0].set_ylabel("Output")
axes[0].set_title(f"VectorizedSCLayer ({N_IN}→{N_OUT}, L={L})")
axes[0].legend()

means = outputs.mean(axis=0)
stds = outputs.std(axis=0)
axes[1].bar(range(N_OUT), means, yerr=stds, capsize=5)
axes[1].set_xlabel("Neuron index")
axes[1].set_ylabel("Mean output ± σ")
axes[1].set_title(f"Output statistics ({N_TRIALS} trials)")
plt.tight_layout()
plt.show()

print(f"Output mean: {means}")
print(f"Output std:  {stds}")

## 6. Accuracy vs Bitstream Length

Stochastic computing trades precision for circuit simplicity.
Accuracy improves as $O(1/\sqrt{L})$ with bitstream length $L$.

In [ ]:
target = 0.65
lengths = [32, 64, 128, 256, 512, 1024, 2048, 4096]
errors = []

for L in lengths:
    enc = BitstreamEncoder(x_min=0.0, x_max=1.0, length=L, seed=42)
    trials = [abs(bitstream_to_probability(enc.encode(target)) - target) for _ in range(200)]
    errors.append(np.mean(trials))

fig, ax = plt.subplots(figsize=(8, 4))
ax.loglog(lengths, errors, "o-", label="Measured MAE")
theoretical = [0.5 / np.sqrt(L) for L in lengths]
ax.loglog(lengths, theoretical, "k--", alpha=0.4, label=r"$0.5 / \sqrt{L}$")
ax.set_xlabel("Bitstream length L")
ax.set_ylabel("Mean absolute error")
ax.set_title("SC Encoding Accuracy vs Bitstream Length")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Summary

| Stage | SC Primitive | Gate Cost |
|-------|-------------|----------|
| Encode | LFSR + comparator | ~20 LUTs |
| Multiply | AND gate | 1 LUT |
| Accumulate | Popcount tree | ~log₂(N) LUTs |
| Integrate | Q8.8 adder + comparator | ~32 LUTs |

The entire neuron pipeline maps to <100 LUTs on a Xilinx Artix-7,
enabling thousands of neurons per FPGA. See `hdl/` for the Verilog RTL
and `scripts/cosim_gen_and_check.py` for bit-exact Python↔Verilog verification.